In [1]:
import pandas as pd
from pathlib import Path
import gc

In [2]:
IEEE_DIR = Path("../../data/raw/ieee-fraud-detection")

In [3]:
train_tx = pd.read_csv(IEEE_DIR / "train_transaction.csv")
id_tx = pd.read_csv(IEEE_DIR / "train_identity.csv").assign(
    _has_identity=True
)


In [4]:
joined = train_tx.join(id_tx.set_index("TransactionID"), on="TransactionID", how="left")
matched = joined["_has_identity"].notna()

print(f"train_tx rows={len(train_tx):,}")
print(f"id_tx rows={len(id_tx):,}  unique TransactionID={id_tx['TransactionID'].nunique():,}")
print(f"matches={matched.sum():,}  match_rate={matched.mean():.2%}")
print(f"unmatched train_tx={ (~matched).sum():,}  miss_rate={(~matched).mean():.2%}")

id_in_tx = id_tx["TransactionID"].isin(train_tx["TransactionID"])
print(f"id_tx keys found in train_tx={id_in_tx.mean():.2%}  ({id_in_tx.sum():,}/{len(id_tx):,})")




train_tx rows=590,540
id_tx rows=144,233  unique TransactionID=144,233
matches=144,233  match_rate=24.42%
unmatched train_tx=446,307  miss_rate=75.58%
id_tx keys found in train_tx=100.00%  (144,233/144,233)


In [ ]:
del train_tx, id_tx
del  matched
gc.collect()

NameError: name 'id_in_tx' is not defined

In [6]:
joined.query("_has_identity == True").groupby("ProductCD").agg({"isFraud": "mean"})

,isFraud
ProductCD,
C,0.122845
H,0.047739
R,0.037898
S,0.059042


## ProductCD nulls

The writeup: W ≈ card-present (addr + `dist1` + M; no recipient email). C ≈ CNP (addr missing ~95%, has `R_emaildomain`, ~12% fraud). H/R/S ≈ digital (no `dist1`; H and R have identity more often).

`query("_has_identity == True")` drops **all of W** — W never joins identity. Check nulls on the full left join first, then the identity slice.


In [7]:
null_cols = [
    "addr1", "addr2", "dist1", "dist2",
    "P_emaildomain", "R_emaildomain",
    "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9",
]

joined["_has_identity"] = joined["_has_identity"].fillna(False)

def not_null_by_product(df):
    return df.groupby("ProductCD")[null_cols].apply(lambda g: 1-g.isnull().mean())

summary = joined.groupby("ProductCD").agg(
    n=("isFraud", "size"),
    fraud=("isFraud", "mean"),
    identity=("_has_identity", "mean"),
)

def style_presence(df):
    return (
        df.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1)
        .format("{:.1%}")
    )

print("full left join")
display(summary)
display(style_presence(not_null_by_product(joined)))

print("identity matched only — W is empty")
id_only = joined.query("_has_identity == True")
display(id_only.groupby("ProductCD").agg(n=("isFraud", "size"), fraud=("isFraud", "mean")))
display(style_presence(not_null_by_product(id_only)))


full left join


,n,fraud,identity
ProductCD,,,
C,68519,0.116873,0.907661
H,33024,0.047662,0.996487
R,37699,0.037826,0.995995
S,11628,0.058996,0.996302
W,439670,0.020399,0.0


,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,M1,M2,M3,M4,M5,M6,M7,M8,M9
ProductCD,,,,,,,,,,,,,,,
C,5.0%,5.0%,0.0%,39.0%,96.9%,96.9%,0.0%,0.0%,0.0%,97.5%,0.0%,0.0%,0.0%,0.0%,0.0%
H,99.7%,99.7%,0.0%,2.7%,100.0%,67.2%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
R,99.9%,99.9%,0.0%,14.7%,99.9%,100.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
S,98.2%,98.2%,0.0%,38.4%,0.0%,94.9%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
W,99.9%,99.9%,54.2%,0.0%,81.7%,0.0%,72.7%,72.7%,72.7%,55.1%,54.6%,95.8%,55.6%,55.6%,55.6%


identity matched only — W is empty


,n,fraud
ProductCD,,
C,62192,0.122845
H,32908,0.047739
R,37548,0.037898
S,11585,0.059042


,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,M1,M2,M3,M4,M5,M6,M7,M8,M9
ProductCD,,,,,,,,,,,,,,,
C,3.4%,3.4%,0.0%,42.9%,97.2%,97.2%,0.0%,0.0%,0.0%,97.8%,0.0%,0.0%,0.0%,0.0%,0.0%
H,99.7%,99.7%,0.0%,2.7%,100.0%,67.2%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
R,99.9%,99.9%,0.0%,14.7%,99.9%,100.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
S,98.2%,98.2%,0.0%,38.5%,0.0%,94.9%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%


In [8]:
del id_only

In [9]:
n = joined.groupby("ProductCD").size()
desc = joined.groupby("ProductCD")[null_cols].describe()
numeric_cols = desc.columns.get_level_values(0).unique()
null_pct = 1 - desc.loc[:, pd.IndexSlice[:, "count"]].droplevel(1, axis=1).div(n, axis=0)
null_block = pd.concat({"null%": null_pct[numeric_cols]}, axis=1).swaplevel(0, 1, axis=1)
out = pd.concat([desc, null_block], axis=1)
stat_order = ["count", "null%", "mean", "std", "min", "25%", "50%", "75%", "max"]
out = out.reindex(columns=pd.MultiIndex.from_product([numeric_cols, stat_order]))
out


addr1                                                         \
              count     null%        mean         std    min    25%    50%   
ProductCD                                                                    
C            3400.0  0.950379  304.045882  135.051380  100.0  161.0  284.0   
H           32940.0  0.002544  294.645902   98.722764  110.0  220.0  299.0   
R           37649.0  0.001326  293.428219   99.368983  110.0  204.0  299.0   
S           11419.0  0.017974  311.721166   86.152625  110.0  299.0  330.0   
W          439426.0  0.000555  289.561303  102.160167  104.0  204.0  299.0   

                            addr2  ...    dist1    dist2            \
             75%    max     count  ...      max    count     null%   
ProductCD                          ...                               
C          465.0  540.0    3400.0  ...      NaN  26741.0  0.609729   
H          330.0  536.0   32940.0  ...      NaN    901.0  0.972717   
R          330.0  536.0   37649.0  ...      NaN   5524.0  0.853471   
S          330.0  536.0   11419.0  ...      NaN   4461.0  0.616357   
W          330.0  536.0  439426.0  ...  10286.0      0.0  1.000000   

                                                                   
                 mean         std  min  25%   50%    75%      max  
ProductCD                                                          
C          224.828765  481.911824  0.0  7.0  60.0  226.0  10237.0  
H          257.541620  646.198201  0.0  7.0  19.0  148.0   7380.0  
R          288.419080  727.350095  0.0  6.0  20.0  149.0  11623.0  
S          198.746021  473.320625  0.0  7.0  30.0  168.0   4475.0  
W                 NaN         NaN  NaN  NaN   NaN    NaN      NaN  

[5 rows x 36 columns]

In [10]:
from IPython.display import HTML, display


display(HTML(f"""
<div style="max-height:500px; overflow:auto;">
    {out.to_html(index=True)}
</div>
"""))

In [ ]:

test_tx = pd.read_csv(IEEE_DIR / "test_transaction.csv")
id_test_tx = pd.read_csv(IEEE_DIR / "test_identity.csv").assign(
    _has_identity=True
)
joined_test = test_tx.join(id_test_tx.set_index("TransactionID"), on="TransactionID", how="left")

del id_test_tx, test_tx
import gc
gc.collect()

In [15]:
%whos

Variable              Type         Data/Info
--------------------------------------------
HTML                  type         <class 'IPython.core.display.HTML'>
IEEE_DIR              PosixPath    ../../data/raw/ieee-fraud-detection
Path                  type         <class 'pathlib._local.Path'>
desc                  DataFrame    Shape: (5, 32)
display               function     <function display at 0x744dda4f8f40>
gc                    module       <module 'gc' (built-in)>
joined                DataFrame    Shape: (590540, 435)
joined_test           DataFrame    Shape: (506691, 434)
n                     Series       Shape: (5,)
not_null_by_product   function     <function not_null_by_product at 0x744d7d9a3380>
null_block            DataFrame    Shape: (5, 4)
null_cols             list         n=15
null_pct              DataFrame    Shape: (5, 4)
numeric_cols          Index        Index(['addr1', 'addr2', <...>', 'dist2'], dtype='str')
out                   DataFrame    Shape: (5, 36)